In [38]:
pip install catboost

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from lightgbm.callback import early_stopping, log_evaluation
from catboost import CatBoostClassifier, Pool

SEED = 42
NFOLD = 5


In [3]:
train = pd.read_csv(r"C:\Users\snarb\Downloads\train.csv")
test = pd.read_csv(r"C:\Users\snarb\Downloads\test.csv")
sample_sub = pd.read_csv(r"C:\Users\snarb\Downloads\sample_submission.csv")


In [5]:
placeholder_map = {365243: np.nan, -1: np.nan}

for col in train.select_dtypes(include=[np.number]).columns:
    train[col] = train[col].replace(placeholder_map)

for col in test.select_dtypes(include=[np.number]).columns:
    test[col] = test[col].replace(placeholder_map)


In [7]:
def feature_engineering(train, test):
    
    if 'AMT_CREDIT' in train.columns and 'AMT_INCOME_TOTAL' in train.columns:
        train['CREDIT_INCOME_RATIO'] = train['AMT_CREDIT'] / (train['AMT_INCOME_TOTAL'] + 1)
        test['CREDIT_INCOME_RATIO'] = test['AMT_CREDIT'] / (test['AMT_INCOME_TOTAL'] + 1)

    if 'AMT_ANNUITY' in train.columns and 'AMT_INCOME_TOTAL' in train.columns:
        train['ANNUITY_INCOME_RATIO'] = train['AMT_ANNUITY'] / (train['AMT_INCOME_TOTAL'] + 1)
        test['ANNUITY_INCOME_RATIO'] = test['AMT_ANNUITY'] / (test['AMT_INCOME_TOTAL'] + 1)

    if 'AMT_CREDIT' in train.columns and 'AMT_ANNUITY' in train.columns:
        train['CREDIT_ANNUITY_RATIO'] = train['AMT_CREDIT'] / (train['AMT_ANNUITY'] + 1)
        test['CREDIT_ANNUITY_RATIO'] = test['AMT_CREDIT'] / (test['AMT_ANNUITY'] + 1)

    if 'DAYS_BIRTH' in train.columns:
        train['AGE_YEARS'] = (-train['DAYS_BIRTH']) / 365.0
        test['AGE_YEARS'] = (-test['DAYS_BIRTH']) / 365.0

    if 'DAYS_EMPLOYED' in train.columns:
        train['EMPLOYED_YEARS'] = train['DAYS_EMPLOYED'].abs() / 365.0
        test['EMPLOYED_YEARS'] = test['DAYS_EMPLOYED'].abs() / 365.0

    # Advanced ratios & differences
    for df in [train, test]:
        if 'AMT_GOODS_PRICE' in df.columns and 'AMT_CREDIT' in df.columns:
            df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + 1)
        if 'CNT_FAM_MEMBERS' in df.columns and 'CNT_CHILDREN' in df.columns:
            df['CHILDREN_RATIO'] = df['CNT_CHILDREN'] / (df['CNT_FAM_MEMBERS'] + 1)
        if 'DAYS_EMPLOYED' in df.columns and 'DAYS_BIRTH' in df.columns:
            df['EMPLOYED_RATIO_TO_AGE'] = df['DAYS_EMPLOYED'] / (df['DAYS_BIRTH'] + 1)
        if 'DAYS_LAST_PHONE_CHANGE' in df.columns and 'DAYS_BIRTH' in df.columns:
            df['PHONE_CHANGE_RATIO'] = df['DAYS_LAST_PHONE_CHANGE'] / (df['DAYS_BIRTH'] + 1)
        if 'EMPLOYED_YEARS' in df.columns and 'AGE_YEARS' in df.columns:
            df['EMPLOYED_AGE_RATIO'] = df['EMPLOYED_YEARS'] / (df['AGE_YEARS'] + 1)
            df['AGE_EMPLOYED_GAP'] = df['AGE_YEARS'] - df['EMPLOYED_YEARS']
            df['ANNUITY_CREDIT_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + 1)

   #--- NEW FEATURES ADDED ---
    for df in [train, test]:
        # EXT_SOURCE interactions (if exist)
        ext_cols = [c for c in ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3'] if c in df.columns]

    for df in [train, test]:
        if len(ext_cols) > 0:
            df['EXT_MEAN'] = df[ext_cols].mean(axis=1)
            df['EXT_MIN'] = df[ext_cols].min(axis=1)
            df['EXT_MAX'] = df[ext_cols].max(axis=1)
            df['EXT_STD'] = df[ext_cols].std(axis=1)
            df['EXT_MISS_CNT'] = df[ext_cols].isnull().sum(axis=1)

        # Additional ratios
        if 'AMT_CREDIT' in df.columns and 'AMT_GOODS_PRICE' in df.columns:
            df['CREDIT_GOODS_DIFF'] = df['AMT_CREDIT'] - df['AMT_GOODS_PRICE']
        if 'AGE_YEARS' in df.columns and 'EXT_MEAN' in df.columns:
            df['AGE_EXT_SOURCE'] = df['AGE_YEARS'] * df['EXT_MEAN']


In [9]:
features = [c for c in train.columns if c not in ['SK_ID_CURR','TARGET']]
X = train[features].copy()
y = train['TARGET'].copy()
X_test = test[features].copy()

# Fill missing with median
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols:
    med = X[col].median()
    X[col] = X[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    mode_val = X[col].mode()[0]
    X[col] = X[col].fillna(mode_val)
    X_test[col] = X_test[col].fillna(mode_val)


In [11]:
all_cat_features = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
                    'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 
                    'OCCUPATION_TYPE_SIMPLE_enc']

for col in all_cat_features:
    if col in X.columns:
        le = LabelEncoder()
        combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
        le.fit(combined)
        X[col] = le.transform(X[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))


In [13]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)


In [15]:
def train_model(X, y, X_test, model_type='lgb'):
    skf = StratifiedKFold(n_splits=NFOLD, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X))
    preds = np.zeros(len(X_test))

    if model_type == 'lgb':
        params = {'objective':'binary','boosting_type':'gbdt','metric':'auc','verbosity':-1,
                  'seed':SEED,'learning_rate':0.03,'num_leaves':63,'max_depth':8,
                  'min_child_samples':30,'subsample':0.8,'subsample_freq':1,'colsample_bytree':0.7,
                  'reg_alpha':0.1,'reg_lambda':2.0,'n_jobs':-1, 'max_bin': 255,'feature_fraction': 0.8,
                  'bagging_fraction': 0.8,'bagging_freq': 1, 'min_gain_to_split': 0.01}
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            train_set = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
            val_set = lgb.Dataset(X_val, label=y_val, reference=train_set, categorical_feature=cat_cols)
            model = lgb.train(params, train_set, num_boost_round=5000,
                              valid_sets=[train_set,val_set],
                              callbacks=[early_stopping(stopping_rounds=80), log_evaluation(200)])
            oof[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
            preds += model.predict(X_test, num_iteration=model.best_iteration)/NFOLD
            del model, train_set, val_set; gc.collect()
    elif model_type == 'catboost':
        cat_idx = [X.columns.get_loc(c) for c in all_cat_features if c in X.columns]
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            train_pool = Pool(X_tr, label=y_tr, cat_features=cat_idx)
            val_pool = Pool(X_val, label=y_val, cat_features=cat_idx)
            model = CatBoostClassifier(iterations=5000, learning_rate=0.04, depth=8,
                                       l2_leaf_reg=6,
                                       bagging_temperature=0.8,
                                       random_strength=1.5,
                                       eval_metric='AUC', random_seed=SEED,
                                       early_stopping_rounds=60, verbose=200)
            model.fit(train_pool, eval_set=val_pool, use_best_model=True)
            oof[val_idx] = model.predict_proba(X_val)[:,1]
            preds += model.predict_proba(X_test)[:,1]/NFOLD
            del model, train_pool, val_pool; gc.collect()
    elif model_type == 'lr':
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = LogisticRegression(max_iter=1000, random_state=SEED,C=0.1,penalty='l2',solver='lbfgs')
            model.fit(X_tr, y_tr)
            oof[val_idx] = model.predict_proba(X_val)[:,1]
            preds += model.predict_proba(X_test_scaled)[:,1]/NFOLD
    elif model_type == 'xgb':
        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'auc',
            'learning_rate': 0.03,
            'max_depth': 6,
            'min_child_weight': 40,
            'subsample': 0.8,
            'colsample_bytree': 0.6,
            'gamma': 0.1,
            'lambda': 2.0,
            'alpha': 0.1,
            'tree_method': 'hist',
            'random_state': SEED,
            'n_jobs': -1
        }

        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

            dtrain = xgb.DMatrix(X_tr, label=y_tr)
            dval = xgb.DMatrix(X_val, label=y_val)
            dtest = xgb.DMatrix(X_test)

            model = xgb.train(
                params,
                dtrain,
                num_boost_round=5000,
                evals=[(dtrain, 'train'), (dval, 'valid')],
                early_stopping_rounds=80,
                verbose_eval=200
            )

            oof[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration))
            preds += model.predict(dtest, iteration_range=(0, model.best_iteration)) / NFOLD

            del model, dtrain, dval, dtest
            gc.collect()

    return oof, preds


In [17]:
oof_lgb, preds_lgb = train_model(X, y, X_test, 'lgb')
print(f"LightGBM OOF AUC: {roc_auc_score(y,oof_lgb):.6f}")

oof_cb, preds_cb = train_model(X, y, X_test, 'catboost')
print(f"CatBoost OOF AUC: {roc_auc_score(y,oof_cb):.6f}")

oof_lr, preds_lr = train_model(X, y, X_test, 'lr')
print(f"Logistic Regression OOF AUC: {roc_auc_score(y,oof_lr):.6f}")

Training until validation scores don't improve for 80 rounds
[200]	training's auc: 0.821398	valid_1's auc: 0.745364
Early stopping, best iteration is:
[253]	training's auc: 0.834712	valid_1's auc: 0.74606
Training until validation scores don't improve for 80 rounds
[200]	training's auc: 0.820985	valid_1's auc: 0.747792
[400]	training's auc: 0.864517	valid_1's auc: 0.749445
Early stopping, best iteration is:
[427]	training's auc: 0.869339	valid_1's auc: 0.749704
Training until validation scores don't improve for 80 rounds
[200]	training's auc: 0.819773	valid_1's auc: 0.752292
Early stopping, best iteration is:
[297]	training's auc: 0.84423	valid_1's auc: 0.753663
Training until validation scores don't improve for 80 rounds
[200]	training's auc: 0.821374	valid_1's auc: 0.743802
[400]	training's auc: 0.866428	valid_1's auc: 0.74535
Early stopping, best iteration is:
[360]	training's auc: 0.858518	valid_1's auc: 0.74556
Training until validation scores don't improve for 80 rounds
[200]	tra

In [19]:
import xgboost as xgb

In [21]:
oof_xgb, preds_xgb = train_model(X, y, X_test, 'xgb')
print(f"XGBoost OOF AUC: {roc_auc_score(y, oof_xgb):.6f}")


[0]	train-auc:0.71294	valid-auc:0.70548
[200]	train-auc:0.77797	valid-auc:0.74625
[400]	train-auc:0.79670	valid-auc:0.74966
[497]	train-auc:0.80403	valid-auc:0.74971
[0]	train-auc:0.71356	valid-auc:0.70789
[200]	train-auc:0.77712	valid-auc:0.74874
[400]	train-auc:0.79556	valid-auc:0.75218
[600]	train-auc:0.80945	valid-auc:0.75316
[740]	train-auc:0.81824	valid-auc:0.75322
[0]	train-auc:0.71319	valid-auc:0.71145
[200]	train-auc:0.77565	valid-auc:0.75307
[400]	train-auc:0.79487	valid-auc:0.75661
[600]	train-auc:0.81018	valid-auc:0.75721
[633]	train-auc:0.81218	valid-auc:0.75738
[0]	train-auc:0.71493	valid-auc:0.70387
[200]	train-auc:0.77785	valid-auc:0.74436
[400]	train-auc:0.79636	valid-auc:0.74678
[600]	train-auc:0.81067	valid-auc:0.74751
[636]	train-auc:0.81283	valid-auc:0.74734
[0]	train-auc:0.71445	valid-auc:0.69904
[200]	train-auc:0.77631	valid-auc:0.74773
[400]	train-auc:0.79469	valid-auc:0.75279
[600]	train-auc:0.80897	valid-auc:0.75369
[800]	train-auc:0.82129	valid-auc:0.75407
[8

In [ ]:
# Stack OOF predictions for meta-model training
oof_stack = np.vstack([oof_lgb, oof_cb]).T

# Stack test predictions for final submission
test_stack = np.vstack([preds_lgb, preds_cb]).T

# Fit meta-model (Logistic Regression)
meta_model = LogisticRegression(C=0.1, penalty='l2', solver='lbfgs', max_iter=1000, random_state=SEED)
meta_model.fit(oof_stack, y)

#Final stacked predictions
final_preds = meta_model.predict_proba(test_stack)[:,1]


In [58]:
import pandas as pd

submission = pd.DataFrame({
    'SK_ID_CURR': test['SK_ID_CURR'],
    'TARGET': final_preds
})

# Save as CSV
submission.to_csv('submission5.csv', index=False)
